# PlusRise Scoring Analysis — TopKonfidence

Consolidated notebook for the analyses in `plusrise_dataset/`. Reproduces the
figures that used to live in separate scripts under `scripts/` (kept there for reference),
now driven by a single **task ID** parameter and rendered inline.

**Folder layout after reorganisation:**
- `scripts/` — original standalone `.py` analysis scripts (unchanged, kept for reference)
- `data/processing/` — cached scored annex summary CSV for the task
- `data/spectra/` — raw MS/MS spectra (MGF) for the task
- `outputs/figures/` — static SVG figures (regenerated by this notebook)
- `outputs/reports/` — text statistics reports
- `outputs/reference/` — original screenshot this analysis reproduces

**Sections:**
1. Parameters & setup
2. Load scored library matches + raw MGF spectra for `TASK_ID`
3. MW vs confidence label (`mw_analysis.py`)
4. Violin: MW by confidence label, with significance testing (`violin_mw_by_label.py`)
5. Tanimoto score vs confidence score (`tanimoto_vs_confidence.py`)
6. Spectral entropy vs confidence (`entropy_analysis.py`)
7. Composite multi-panel figure (`composite_figure.py`)
8. `top_compound` vs `top_cosine_compound` comparison report (`compare_top_compounds.py`)


## 1. Parameters & setup

In [ ]:
import sys, os, re, math
import warnings, logging

logging.disable(logging.CRITICAL)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from bin.mgf_loader import _parse_mgf
from bin.spectral_entropy_v2 import (
    DEFAULT_MAX_PEAK_NUM, DEFAULT_MS2_TOLERANCE_DA, DEFAULT_NOISE_THRESHOLD,
    spectral_entropy_for_spectrum,
)
from bin.normalizer import _normalize_library_matches_dataframe
from bin.scorer import _compute_scan_level_confidence

BASE     = os.getcwd()                     # plusrise_dataset
DATA     = os.path.join(BASE, "data")
PROC     = os.path.join(DATA, "processing")
SPECTRA  = os.path.join(DATA, "spectra")
OUT_FIG  = os.path.join(BASE, "outputs", "figures")
OUT_REP  = os.path.join(BASE, "outputs", "reports")
os.makedirs(OUT_FIG, exist_ok=True)
os.makedirs(OUT_REP, exist_ok=True)

print("Repo root:", REPO_ROOT)


### Parameters

`TASK_ID` is the GNPS2 FBMN task whose scored library-match results (`annex`) and raw MGF
spectra drive every figure. Defaults to the test FBMN task documented in the project's
`CLAUDE.md`. Set `USE_CACHED_DATA = False` to force a live re-fetch + re-score from GNPS2
instead of the cached files under `data/processing/` and `data/spectra/`.

In [ ]:
# ── notebook parameters ────────────────────────────────────────────────────
TASK_ID         = "b22ff1bc41624e29a28cf6e12fe17ad5"
WORKFLOW_CHOICE = "fbmn"
USE_CACHED_DATA = True   # reuse data/processing + data/spectra cache instead of hitting GNPS2 live

PROTON = 1.007276

LABEL_ORDER = ["Consistent evidence", "Inconclusive", "Inconsistent evidence"]
LABEL_COLOR = {
    "Consistent evidence":  "#008300",
    "Inconclusive":          "#eda100",
    "Inconsistent evidence": "#e34948",
}
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#dcdbd6", "#ffffff"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 10, "text.color": INK, "axes.labelcolor": INK_2, "axes.edgecolor": GRID,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
})


def save_and_show(fig, name):
    svg = os.path.join(OUT_FIG, f"{name}.svg")
    fig.savefig(svg, dpi=200, bbox_inches="tight")
    print(f"Saved -> {svg}")
    plt.show()


## 2. Load scored library matches + raw MGF spectra for `TASK_ID`

`annex` mirrors `annex_summary_<hash>.csv` (one row per scan, confidence score + component
breakdown). It's read from the on-disk cache when available, otherwise fetched + scored
live from GNPS2 (same loader chain as the Flask app: `taskresult` TSV → `workflow_fbmn`
fallback → `_normalize_library_matches_dataframe` → `_compute_scan_level_confidence`).

`raw_matches` mirrors the full per-hit library-match rows (needed only for §8, the
top-compound vs top-cosine-compound comparison) — same fetch chain, kept unnormalized.

In [ ]:
def fetch_raw_matches(task_id):
    """Raw (unnormalized) library-match rows for task_id — one row per hit."""
    raw_df = None
    try:
        from gnpsdata import taskresult
        raw_df = taskresult.get_gnps2_task_resultfile_dataframe(
            task_id, "nf_output/library/merged_results_with_gnps.tsv"
        )
        if raw_df is None or len(raw_df) == 0:
            raw_df = None
        else:
            print(f"  Loaded {len(raw_df)} rows via taskresult TSV")
    except Exception as exc:
        print(f"  taskresult TSV failed: {exc}")

    if raw_df is None:
        try:
            from gnpsdata import workflow_fbmn
            raw_df = workflow_fbmn.get_library_match_dataframe(task_id, gnps2=True)
            if raw_df is None or len(raw_df) == 0:
                raw_df = None
            else:
                print(f"  Loaded {len(raw_df)} rows via workflow_fbmn")
        except Exception as exc:
            print(f"  workflow_fbmn failed: {exc}")

    if raw_df is None:
        raise RuntimeError(f"Could not load library matches for task {task_id}")
    return raw_df


def load_annex(task_id, workflow_choice=WORKFLOW_CHOICE, use_cache=USE_CACHED_DATA):
    """Scored per-scan summary for task_id (annex_summary_<hash>.csv equivalent)."""
    cache_hash = task_id[:12]
    cached = os.path.join(PROC, f"annex_summary_{cache_hash} (6).csv")
    if not os.path.exists(cached):
        cached = os.path.join(PROC, f"annex_summary_{cache_hash}.csv")

    if use_cache and os.path.exists(cached):
        print(f"Using cached annex summary for task {task_id}")
        annex = pd.read_csv(cached)
    else:
        print(f"Fetching + scoring task {task_id} live from GNPS2 ...")
        raw_df = fetch_raw_matches(task_id)
        norm_df = _normalize_library_matches_dataframe(raw_df)
        annex = _compute_scan_level_confidence(norm_df, task_id=task_id, workflow_choice=workflow_choice)

    annex["scan"] = annex["scan"].astype(int)
    return annex


annex = load_annex(TASK_ID)
print(f"annex: {len(annex)} scored scans")
annex.head()


In [ ]:
# ── raw MGF spectra ─────────────────────────────────────────────────────────
MGF_PATH = None
if USE_CACHED_DATA:
    for fname in os.listdir(SPECTRA):
        if TASK_ID in fname and fname.endswith(".mgf"):
            MGF_PATH = os.path.join(SPECTRA, fname)
            break

if MGF_PATH is None:
    raise FileNotFoundError(
        f"No cached MGF found for task {TASK_ID} under {SPECTRA}. "
        "Live MGF fetch is not wired up in this notebook — supply a local MGF "
        "(e.g. via the FBMN task's spectra_reformatted.mgf) to proceed."
    )

print("Using MGF:", MGF_PATH)


def parse_mgf_df(path):
    """Return DataFrame: scan, mgf_pepmass, charge, n_peaks (mirrors mw_analysis.parse_mgf)."""
    peak_re = re.compile(r"^\d")
    rows, cur = [], None
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line == "BEGIN IONS":
                cur = {"scan": None, "mgf_pepmass": np.nan, "charge": np.nan, "n_peaks": 0}
            elif line == "END IONS":
                if cur and cur["scan"] is not None:
                    rows.append(cur)
                cur = None
            elif cur is None:
                continue
            elif line.startswith("SCANS="):
                cur["scan"] = int(line[6:])
            elif line.startswith("PEPMASS="):
                cur["mgf_pepmass"] = float(line[8:].split()[0])
            elif line.startswith("CHARGE="):
                cur["charge"] = float(re.sub(r"[^0-9.\-+]", "", line[7:]) or "nan")
            elif peak_re.match(line):
                cur["n_peaks"] += 1
    return pd.DataFrame(rows)


def neutral_mass(mz, charge):
    z = np.where((charge >= 1) & np.isfinite(charge), charge, 1.0)
    return mz * z - z * PROTON


spectra_df = parse_mgf_df(MGF_PATH)
print(f"Parsed {len(spectra_df)} spectra from MGF")


## 3. MW vs confidence label

Two figures: (1) confidence-label distribution + median-score trend across precursor MW
bins, (2) MW vs MS/MS peak count. `mw` is neutral monoisotopic mass derived from the
precursor m/z and charge (MGF `CHARGE=` field, clamped to 1–6, defaulting to 1).

In [ ]:
def build_mw_frame():
    df = annex.merge(spectra_df, on="scan", how="left")
    df["mz"] = df["precursor_mz"].fillna(df["mgf_pepmass"])
    df["charge"] = df["charge"].where(df["charge"].between(1, 6), 1.0)
    df["mw"] = neutral_mass(df["mz"].to_numpy(), df["charge"].to_numpy())
    df = df[df["mw"].between(50, 2000) & df["confidence_label"].isin(LABEL_ORDER)].copy()
    df["n_peaks"] = df["n_peaks"].fillna(df["query_peak_count"])
    return df


df3 = build_mw_frame()
print(f"scans scored: {len(annex)}   matched to MGF: {df3['n_peaks'].notna().sum()}   analysed: {len(df3)}")
print(f"charge states used: {df3['charge'].value_counts().to_dict()}")

MW_EDGES = [0, 150, 200, 250, 300, 350, 400, 500, 600, 10000]
MW_LABELS = ["<150", "150–200", "200–250", "250–300", "300–350", "350–400", "400–500", "500–600", "≥600"]
df3["mw_bin"] = pd.cut(df3["mw"], bins=MW_EDGES, labels=MW_LABELS, right=False)

ct = pd.crosstab(df3["mw_bin"], df3["confidence_label"]).reindex(columns=LABEL_ORDER, fill_value=0)
frac = ct.div(ct.sum(axis=1), axis=0) * 100
n_per_bin = ct.sum(axis=1)


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), height_ratios=[2.2, 1], sharex=True,
                                gridspec_kw={"hspace": 0.18})

x = np.arange(len(frac))
bottom = np.zeros(len(frac))
for lab in LABEL_ORDER:
    vals = frac[lab].to_numpy()
    ax1.bar(x, vals, bottom=bottom, width=0.72, color=LABEL_COLOR[lab], label=lab, edgecolor=SURFACE, linewidth=2)
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v >= 6:
            ax1.text(xi, b + v / 2, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    bottom += vals
for xi, n in enumerate(n_per_bin):
    ax1.text(xi, 101.5, f"n={n}", ha="center", va="bottom", fontsize=8, color=INK_2)
ax1.set_ylim(0, 108)
ax1.set_ylabel("Share of scans (%)")
ax1.set_title("Confidence label distribution by precursor molecular weight", fontsize=13, fontweight="bold", loc="left", pad=26)
ax1.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.005), fontsize=9)
ax1.grid(axis="y", color=GRID, linewidth=0.8); ax1.set_axisbelow(True)

med = df3.groupby("mw_bin", observed=True)["confidence_score"].median().reindex(MW_LABELS)
q1 = df3.groupby("mw_bin", observed=True)["confidence_score"].quantile(0.25).reindex(MW_LABELS)
q3 = df3.groupby("mw_bin", observed=True)["confidence_score"].quantile(0.75).reindex(MW_LABELS)
ax2.fill_between(x, q1, q3, color="#2a78d6", alpha=0.16, linewidth=0)
ax2.plot(x, med, color="#2a78d6", linewidth=2, marker="o", markersize=7,
         markeredgecolor=SURFACE, markeredgewidth=1.5, label="median score (IQR band)")
ax2.axhline(80, color=INK_2, linewidth=1, linestyle=":", zorder=0)
ax2.text(len(x) - 0.4, 80.8, "80 = consistent", fontsize=8, color=INK_2, ha="right")
ax2.set_ylabel("Confidence score")
ax2.set_xlabel("Neutral monoisotopic mass (Da)")
ax2.set_xticks(x); ax2.set_xticklabels(MW_LABELS)
ax2.legend(frameon=False, fontsize=9, loc="lower right")
ax2.grid(axis="y", color=GRID, linewidth=0.8); ax2.set_axisbelow(True)

save_and_show(fig, "fig_mw_vs_confidence")


In [ ]:
d2 = df3.dropna(subset=["n_peaks"])
fig, ax = plt.subplots(figsize=(9, 6))
for lab in LABEL_ORDER:
    s = d2[d2["confidence_label"] == lab]
    ax.scatter(s["mw"], s["n_peaks"], s=26, alpha=0.55, color=LABEL_COLOR[lab],
               edgecolor=SURFACE, linewidth=0.5, label=f"{lab} (n={len(s)})")

lo = stats.linregress(d2["mw"], d2["n_peaks"])
xs = np.linspace(d2["mw"].min(), d2["mw"].max(), 100)
ax.plot(xs, lo.intercept + lo.slope * xs, color=INK, linewidth=2, linestyle="--", zorder=5, label="linear fit")

mb = pd.cut(d2["mw"], bins=MW_EDGES, labels=MW_LABELS, right=False)
bmed = d2.groupby(mb, observed=True)["n_peaks"].median().reindex(MW_LABELS)
bx = d2.groupby(mb, observed=True)["mw"].median().reindex(MW_LABELS)
ok = bmed.notna() & bx.notna()
ax.plot(bx[ok], bmed[ok], color=INK, linewidth=2.5, marker="o", markersize=9,
        markeredgecolor=SURFACE, markeredgewidth=2, zorder=6, label="median per MW bin")

r_p, p_p = stats.pearsonr(d2["mw"], d2["n_peaks"])
r_s, p_s = stats.spearmanr(d2["mw"], d2["n_peaks"])
ax.text(0.02, 0.97, f"Pearson r = {r_p:.2f}  (p = {p_p:.1e})\nSpearman ρ = {r_s:.2f}  (p = {p_s:.1e})\nslope = {lo.slope:.3f} peaks/Da",
        transform=ax.transAxes, va="top", ha="left", fontsize=9, color=INK_2,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=SURFACE, edgecolor=GRID))
ax.set_xlabel("Neutral monoisotopic mass (Da)")
ax.set_ylabel("Peaks in MS/MS spectrum")
ax.set_title(f"Peak count vs precursor mass — weak positive trend (R² = {lo.rvalue ** 2:.2f})", fontsize=13, fontweight="bold", loc="left", pad=40)
ax.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.005), fontsize=9)
ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)
save_and_show(fig, "fig_mw_vs_peaks")


In [ ]:
out = []
out.append(f"scans scored: {len(annex)}   matched to MGF: {df3['n_peaks'].notna().sum()}   analysed: {len(df3)}")
out.append(f"charge states used: {df3['charge'].value_counts().to_dict()}")
out.append("")
out.append("== MW vs confidence score ==")
rs, ps = stats.spearmanr(df3["mw"], df3["confidence_score"])
rp, pp = stats.pearsonr(df3["mw"], df3["confidence_score"])
out.append(f"Spearman rho = {rs:.3f}  p = {ps:.3e}")
out.append(f"Pearson  r   = {rp:.3f}  p = {pp:.3e}")
groups = [g["mw"].to_numpy() for _, g in df3.groupby("confidence_label") if len(g) > 1]
if len(groups) > 1:
    h, ph = stats.kruskal(*groups)
    out.append(f"Kruskal-Wallis on MW across labels: H = {h:.2f}  p = {ph:.3e}")
out.append("")
out.append("median MW per label:")
out.append(df3.groupby("confidence_label")["mw"].describe()[["count", "25%", "50%", "75%"]].to_string())
out.append("")
out.append("== label counts per MW bin ==")
out.append(ct.to_string())
out.append("")
out.append("== label share (%) per MW bin ==")
out.append(frac.round(1).to_string())
out.append("")
out.append("== confidence score per MW bin ==")
out.append(df3.groupby("mw_bin", observed=True)["confidence_score"].agg(["count", "mean", "median"]).round(2).to_string())
out.append("")
out.append("== MW vs peak count ==")
out.append(f"Pearson  r   = {r_p:.3f}  p = {p_p:.3e}")
out.append(f"Spearman rho = {r_s:.3f}  p = {p_s:.3e}")
out.append(f"linear fit: n_peaks = {lo.slope:.4f} * MW + {lo.intercept:.2f}  (R^2 = {lo.rvalue**2:.3f})")
out.append("")
out.append("== peak count vs confidence score ==")
rs2, ps2 = stats.spearmanr(d2["n_peaks"], d2["confidence_score"])
out.append(f"Spearman rho = {rs2:.3f}  p = {ps2:.3e}")

text = "\n".join(out)
(open(os.path.join(OUT_REP, "mw_analysis_stats.txt"), "w")).write(text + "\n")
print(text)


## 4. Violin: MW by confidence label, with significance testing

Is confidence label related to molecular weight? MW is right-skewed, so the primary test
is non-parametric:

1. Shapiro-Wilk per group — normality check
2. Levene (Brown-Forsythe) — variance homogeneity check
3. **Kruskal-Wallis** — primary omnibus test, + epsilon² effect size
4. Welch ANOVA on log10(MW) — parametric cross-check
5. **Dunn post-hoc, Holm-adjusted** — primary post-hoc (pairs with Kruskal-Wallis)
6. Conover-Iman + Dwass-Steel-Critchlow-Fligner — post-hoc robustness checks
7. Pairwise Mann-Whitney + Holm — kept for Cliff's delta effect sizes
8. Jonckheere-Terpstra — ordered-alternative trend test, validated against a permutation null

In [ ]:
import scikit_posthocs as sp
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway

ORDER = ["Inconsistent evidence", "Inconclusive", "Consistent evidence"]  # ascending confidence


def holm(pvals):
    return multipletests(pvals, method="holm")[1]


def cliffs_delta(a, b):
    """P(a>b) - P(a<b); |d| < .147 negligible, < .33 small, < .474 medium."""
    a, b = np.asarray(a), np.asarray(b)
    u, _ = stats.mannwhitneyu(a, b, alternative="two-sided")
    return 2.0 * u / (len(a) * len(b)) - 1.0


def epsilon_squared(H, n, k):
    """Effect size for Kruskal-Wallis. .01 small, .08 medium, .26 large."""
    return (H - k + 1) / (n - k)


def welch_anova(groups):
    r = anova_oneway(groups, use_var="unequal", welch_correction=True)
    return r.statistic, r.df[0], r.df[1], r.pvalue


def jonckheere_terpstra_perm(groups, n_perm=20000, seed=0):
    def J_fast(parts):
        total = 0.0
        srt = [np.sort(p) for p in parts]
        for i in range(len(parts) - 1):
            for j in range(i + 1, len(parts)):
                lo = np.searchsorted(srt[i], srt[j], side="left")
                hi = np.searchsorted(srt[i], srt[j], side="right")
                total += (lo + hi).sum() / 2.0
        return total

    rng = np.random.default_rng(seed)
    sizes = np.array([len(g) for g in groups])
    bounds = np.cumsum(sizes)[:-1]
    pooled = np.concatenate(groups)
    J_obs = J_fast(groups)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        if J_fast(np.split(pooled, bounds)) >= J_obs:
            count += 1
    return (count + 1) / (n_perm + 1)


def jonckheere_terpstra(groups):
    k = len(groups)
    n = np.array([len(g) for g in groups], float)
    N = n.sum()
    J = 0.0
    for i in range(k - 1):
        for j in range(i + 1, k):
            u, _ = stats.mannwhitneyu(groups[j], groups[i], alternative="two-sided")
            J += u
    mu = (N ** 2 - (n ** 2).sum()) / 4.0
    allv = np.concatenate(groups)
    _, tcnt = np.unique(allv, return_counts=True)
    t = tcnt.astype(float)
    var = (
        (N * (N - 1) * (2 * N + 5) - (n * (n - 1) * (2 * n + 5)).sum() - (t * (t - 1) * (2 * t + 5)).sum()) / 72.0
        + (n * (n - 1) * (n - 2)).sum() * (t * (t - 1) * (t - 2)).sum() / (36.0 * N * (N - 1) * (N - 2))
        + (n * (n - 1)).sum() * (t * (t - 1)).sum() / (8.0 * N * (N - 1))
    )
    z = (J - mu) / np.sqrt(var)
    return J, z, 2 * stats.norm.sf(abs(z))


def build_violin_frame():
    return build_mw_frame()  # same charge-aware MW derivation as section 3


df4 = build_violin_frame()


In [ ]:
groups = [df4.loc[df4["confidence_label"] == lab, "mw"].to_numpy() for lab in ORDER]
out = []

out.append("== assumption checks ==")
for lab, g in zip(ORDER, groups):
    w, pw = stats.shapiro(g)
    wl, pwl = stats.shapiro(np.log10(g))
    out.append(f"  Shapiro-Wilk {lab:24s} n={len(g):4d}  raw W={w:.3f} p={pw:.2e}   log10 W={wl:.3f} p={pwl:.2e}")
lev, plev = stats.levene(*groups, center="median")
out.append(f"  Levene (Brown-Forsythe) W={lev:.3f}  p={plev:.3e}")
out.append("  -> MW is non-normal in every group; Kruskal-Wallis is the primary test.")
out.append("")

H, pH = stats.kruskal(*groups)
eps2 = epsilon_squared(H, len(df4), len(groups))
out.append("== 1. omnibus: Kruskal-Wallis (PRIMARY) ==")
out.append(f"  H = {H:.3f}   df = {len(groups) - 1}   p = {pH:.3e}")
out.append(f"  epsilon^2 = {eps2:.4f}  ({'negligible' if eps2 < 0.01 else 'small' if eps2 < 0.08 else 'medium'})")
out.append("")

F, df1, df2, pF = welch_anova([np.log10(g) for g in groups])
out.append("== 2. cross-check: Welch ANOVA on log10(MW) ==")
out.append(f"  F({df1:.0f}, {df2:.1f}) = {F:.3f}   p = {pF:.3e}")
out.append("")

pairs = [(0, 1), (1, 2), (0, 2)]
dunn = sp.posthoc_dunn(df4, val_col="mw", group_col="confidence_label", p_adjust="holm").reindex(index=ORDER, columns=ORDER)
conover = sp.posthoc_conover(df4, val_col="mw", group_col="confidence_label", p_adjust="holm").reindex(index=ORDER, columns=ORDER)
dscf = sp.posthoc_dscf(df4, val_col="mw", group_col="confidence_label").reindex(index=ORDER, columns=ORDER)

out.append("== 3. post-hoc: Dunn test, Holm-adjusted (PRIMARY) ==")
out.append(dunn.round(5).to_string())
out.append("")
out.append("  robustness checks (same pairs, different post-hoc):")
out.append("  Conover-Iman, Holm-adjusted:")
out.append(conover.round(5).to_string())
out.append("  Dwass-Steel-Critchlow-Fligner (self-adjusting):")
out.append(dscf.round(5).to_string())
out.append("")

raw_p, rows = [], []
for i, j in pairs:
    u, p = stats.mannwhitneyu(groups[i], groups[j], alternative="two-sided")
    d = cliffs_delta(groups[j], groups[i])
    raw_p.append(p)
    rows.append((ORDER[i], ORDER[j], u, p, d))
mw_holm = holm(raw_p)
adj = np.array([dunn.loc[ORDER[i], ORDER[j]] for i, j in pairs])

out.append("== 4. effect sizes (Mann-Whitney U + Cliff's delta) ==")
out.append(f"  {'comparison':52s} {'U':>10s} {'p_holm':>10s} {'p_dunn':>10s} {'Cliff d':>9s}  magnitude")
for (a, b, u, p, d), pm, pd_ in zip(rows, mw_holm, adj):
    mag = ("negligible" if abs(d) < .147 else "small" if abs(d) < .33 else "medium" if abs(d) < .474 else "large")
    sig = "***" if pd_ < .001 else "**" if pd_ < .01 else "*" if pd_ < .05 else "ns"
    out.append(f"  {a + ' vs ' + b:52s} {u:10.0f} {pm:10.2e} {pd_:10.2e} {d:+9.3f}  {mag} {sig}")
out.append("")

J, z, pJ = jonckheere_terpstra(groups)
pJ_perm = jonckheere_terpstra_perm(groups)
out.append("== 5. ordered-alternative trend (Jonckheere-Terpstra) ==")
out.append("  H1: MW increases across Inconsistent -> Inconclusive -> Consistent")
out.append(f"  J = {J:.0f}   z = {z:.3f}   p_normal = {pJ:.3e}")
out.append(f"  permutation p (20k one-sided) = {pJ_perm:.5f}  -> normal approximation validated")
rho, prho = stats.spearmanr(df4["confidence_label"].map({l: i for i, l in enumerate(ORDER)}), df4["mw"])
out.append(f"  Spearman(label rank, MW) rho = {rho:+.3f}  p = {prho:.3e}")
out.append("")

out.append("== descriptives (MW, Da) ==")
desc = df4.groupby("confidence_label")["mw"].agg(
    n="count", mean="mean", sd="std", q1=lambda s: s.quantile(.25),
    median="median", q3=lambda s: s.quantile(.75)).reindex(ORDER).round(1)
out.append(desc.to_string())
out.append("")
out.append("== interpretation ==")
out.append(f"  Groups differ significantly (p = {pH:.2e}) BUT epsilon^2 = {eps2:.4f} means")
out.append(f"  MW explains ~{eps2 * 100:.1f}% of the variance in confidence label.")
out.append("  The relationship is real and monotonic but far too weak to predict a label from mass.")

text = "\n".join(out)
open(os.path.join(OUT_REP, "violin_mw_stats.txt"), "w").write(text + "\n")
print(text)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6.5))
pos = np.arange(1, len(ORDER) + 1)
parts = ax.violinplot(groups, positions=pos, widths=0.78, showmeans=False, showmedians=False, showextrema=False)
for body, lab in zip(parts["bodies"], ORDER):
    body.set_facecolor(LABEL_COLOR[lab]); body.set_alpha(0.32)
    body.set_edgecolor(LABEL_COLOR[lab]); body.set_linewidth(1.5)

bp = ax.boxplot(groups, positions=pos, widths=0.13, patch_artist=True, showfliers=False,
                 medianprops=dict(color=SURFACE, linewidth=2),
                 whiskerprops=dict(color=INK_2, linewidth=1.2), capprops=dict(color=INK_2, linewidth=1.2))
for patch, lab in zip(bp["boxes"], ORDER):
    patch.set_facecolor(LABEL_COLOR[lab]); patch.set_edgecolor(LABEL_COLOR[lab])

rng4 = np.random.default_rng(0)
for p, g, lab in zip(pos, groups, ORDER):
    ax.scatter(p + rng4.uniform(-0.06, 0.06, len(g)), g, s=5, alpha=0.18, color=LABEL_COLOR[lab], linewidth=0, zorder=1)
    ax.text(p + 0.115, np.median(g), f"med {np.median(g):.0f}", va="center", ha="left", fontsize=8.5, color=INK_2)

top = max(g.max() for g in groups)
y0, step = top + 60, 85
for lvl, ((i, j), pa) in enumerate(zip(pairs, adj)):
    y = y0 + lvl * step
    x1, x2 = pos[i], pos[j]
    ax.plot([x1, x1, x2, x2], [y, y + 22, y + 22, y], color=INK_2, linewidth=1.1)
    star = ("***" if pa < .001 else "**" if pa < .01 else "*" if pa < .05 else "ns")
    ptxt = f"p = {pa:.1e}" if pa < 0.001 else f"p = {pa:.3f}"
    ax.text((x1 + x2) / 2, y + 30, f"{star}   {ptxt}", ha="center", fontsize=8.5, color=INK_2)
for p, g in zip(pos, groups):
    ax.text(p, top + 18, f"n = {len(g)}", ha="center", fontsize=9, color=INK_2)

ax.set_xlim(0.45, len(ORDER) + 0.72)
ax.set_ylim(0, y0 + (len(pairs) - 1) * step + 95)
ax.set_xticks(pos)
ax.set_xticklabels([l.replace(" evidence", "\nevidence") for l in ORDER])
ax.set_ylabel("Neutral monoisotopic mass (Da)")
ax.set_xlabel("Confidence label")
ax.set_title("Precursor mass by confidence label", fontsize=13, fontweight="bold", loc="left", pad=44)
ax.text(0, 1.008, f"Kruskal-Wallis H = {H:.1f}, p = {pH:.1e}, ε² = {eps2:.3f} (small) · Jonckheere trend z = {z:.2f}, p = {pJ:.1e}\n"
                  f"brackets: Dunn post-hoc, Holm-adjusted",
        transform=ax.transAxes, fontsize=9, color=INK_2, va="bottom")
ax.grid(axis="y", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
save_and_show(fig, "fig_violin_mw_by_label")


## 5. Tanimoto score vs confidence score

x = `tanimoto_score`, y = `confidence_score`. Marker size & color = number of library hits
(`total_matches`); marker shape = basis of the Tanimoto value (`tanimoto_source`): circle =
measured ECFP4 (≥2 unique structures), diamond = single structure (Tanimoto = 1 by
definition), X = no structure available at all (Tanimoto falls back to support fraction).
Red ring = scan where ≥50% of hits are missing a structure.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

RED = "#e34948"
HIT_CMAP = LinearSegmentedColormap.from_list("hits_blue", ["#bcd4f2", "#2a78d6", "#0b3b73"])
SOURCE_STYLE = {
    "tanimoto_wcrs":                ("o", "Measured ECFP4 (≥2 structures)", 0),
    "single_structure":             ("D", "Single structure (Tanimoto = 1\nby definition)", 0),
    "structure_agreement_fallback": ("X", "No structure available\n(Tanimoto = support fraction)", 55),
}


def hit_size(hits):
    return 22 + (np.asarray(hits, float) - 1) * 20  # 22 .. 202


df5 = annex.dropna(subset=["tanimoto_score", "confidence_score"]).copy()
df5["missing_frac"] = df5["missing_structures"] / df5["total_matches"]
flagged = df5["missing_frac"] >= 0.5
no_structure = df5["missing_structures"] == df5["total_matches"]

lo5 = stats.linregress(df5["tanimoto_score"], df5["confidence_score"])
r5 = lo5.rvalue

fig, ax = plt.subplots(figsize=(9.5, 6.5))
norm = Normalize(1, 10)

for src, (marker, _, bump) in SOURCE_STYLE.items():
    shape_mask = df5["tanimoto_source"] == src
    for is_flag, edge, lw, al in [(False, SURFACE, 0.4, 0.72), (True, RED, 1.6, 0.9)]:
        s = df5[shape_mask & (flagged if is_flag else ~flagged)]
        if s.empty:
            continue
        z = 7 if marker == "X" else (4 if is_flag else 2)
        ax.scatter(s["tanimoto_score"], s["confidence_score"], marker=marker,
                   s=hit_size(s["total_matches"]) + bump, c=s["total_matches"],
                   cmap=HIT_CMAP, norm=norm, alpha=al, edgecolor=edge, linewidth=lw, zorder=z)

x = df5["tanimoto_score"].to_numpy(); y = df5["confidence_score"].to_numpy()
n = len(x)
xs = np.linspace(x.min(), x.max(), 200)
yhat = lo5.intercept + lo5.slope * xs
resid = y - (lo5.intercept + lo5.slope * x)
s_err = np.sqrt(np.sum(resid ** 2) / (n - 2))
sxx = np.sum((x - x.mean()) ** 2)
tval = stats.t.ppf(0.975, n - 2)
ci = tval * s_err * np.sqrt(1.0 / n + (xs - x.mean()) ** 2 / sxx)
ax.fill_between(xs, yhat - ci, yhat + ci, color=RED, alpha=0.14, linewidth=0, zorder=3)
ax.plot(xs, yhat, color=RED, linewidth=2.2, linestyle="--", zorder=5)

sm = plt.cm.ScalarMappable(cmap=HIT_CMAP, norm=norm)
cbar = fig.colorbar(sm, ax=ax, pad=0.015, fraction=0.045)
cbar.set_label("Library hits (n)", color=INK_2)
cbar.set_ticks([1, 2, 4, 6, 8, 10])
cbar.outline.set_edgecolor(GRID); cbar.ax.tick_params(color=GRID)

shape_handles = [
    Line2D([0], [0], marker=mk, linestyle="none", markerfacecolor="#7aa9e0",
           markeredgecolor=(RED if mk == "X" else SURFACE), markeredgewidth=(1.6 if mk == "X" else 0.6),
           markersize=(10 if mk == "X" else 8.5), label=lbl)
    for mk, lbl, _ in SOURCE_STYLE.values()
]
shape_handles.append(Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#7aa9e0",
                             markeredgecolor=RED, markeredgewidth=1.6, markersize=9, label="≥ 50% missing structures"))
leg1 = ax.legend(handles=shape_handles, loc="upper left", frameon=True, facecolor=SURFACE, edgecolor=GRID,
                 framealpha=0.92, fontsize=8.5, labelspacing=1.0, borderpad=0.9, handletextpad=1.0)
leg1.set_zorder(6); ax.add_artist(leg1)

size_handles = [
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor="#7aa9e0", markeredgecolor=SURFACE,
           markersize=np.sqrt(hit_size(h)), label=f"{h} hit" + ("" if h == 1 else "s"))
    for h in (1, 4, 7, 10)
]
leg2 = ax.legend(handles=size_handles, loc="lower right", frameon=True, facecolor=SURFACE, edgecolor=GRID,
                 framealpha=0.92, fontsize=8.5, labelspacing=1.2, borderpad=0.9, handletextpad=1.0, title="Library hits")
leg2.get_title().set_color(INK_2); leg2.set_zorder(6)

ax.set_xlabel("Tanimoto score"); ax.set_ylabel("Confidence score")
ax.set_title("Tanimoto score vs confidence", fontsize=13, fontweight="bold", loc="left", pad=26)
ax.text(0, 1.012, f"OLS r = {r5:.2f} (p = {lo5.pvalue:.1e}, n = {n}) · size & color = library hits · "
                  f"✕ = no structure available (n = {int(no_structure.sum())})",
        transform=ax.transAxes, fontsize=9, color=INK_2, va="bottom")
ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)

save_and_show(fig, "fig_tanimoto_vs_confidence")
print(f"r={r5:.3f}, n={n}, flagged={int(flagged.sum())}")


## 6. Spectral entropy vs confidence

Companion to §3. Instead of the raw MGF peak count (which counts noise peaks), this uses
`ms_entropy` spectral entropy S — computed through the project's own
`bin/spectral_entropy_v2.spectral_entropy_for_spectrum`, so the preprocessing
(`clean_spectrum=True`, 0.01 base-peak noise floor, 0.02 Da peak merging, 100-peak cap)
matches what the dashboard scores against. Three complexity measures compared:
`n_peaks_raw` (every MGF peak), `n_peaks_filtered` (after the scorer's noise floor + cap),
`eff_peaks = e^S` (perplexity — the "effective" number of peaks carrying signal).

In [ ]:
ENTROPY_BANDS = [
    ("noisy / low quality", 0.0, 0.5),
    ("moderate quality", 0.5, 1.0),
    ("clean / good quality", 1.0, 1.75),
    ("very diffuse / uniform", 1.75, np.inf),
]


def build_entropy_frame():
    spectra_raw = _parse_mgf(MGF_PATH)
    rows = []
    for scan_str, peaks in spectra_raw.items():
        if not peaks:
            rows.append({"scan": int(scan_str), "S": np.nan, "n_peaks_raw": 0, "n_peaks_filtered": 0})
            continue
        S = spectral_entropy_for_spectrum(
            peaks, ms2_tolerance_da=DEFAULT_MS2_TOLERANCE_DA,
            noise_threshold=DEFAULT_NOISE_THRESHOLD, max_peak_num=DEFAULT_MAX_PEAK_NUM,
        )
        base = max(p[1] for p in peaks)
        filt = [p for p in peaks if base > 0 and p[1] >= DEFAULT_NOISE_THRESHOLD * base]
        rows.append({"scan": int(scan_str), "S": S, "n_peaks_raw": len(peaks),
                     "n_peaks_filtered": min(len(filt), DEFAULT_MAX_PEAK_NUM)})

    df = annex.merge(pd.DataFrame(rows), on="scan", how="left")
    df["eff_peaks"] = np.exp(df["S"])
    df["mw"] = neutral_mass(df["precursor_mz"].to_numpy(), np.ones(len(df)))  # 99.4% singly charged
    df = df[df["mw"].between(50, 2000) & df["confidence_label"].isin(LABEL_ORDER) & df["S"].notna()].copy()
    df["entropy_band"] = pd.cut(df["S"], bins=[b[1] for b in ENTROPY_BANDS] + [np.inf],
                                 labels=[b[0] for b in ENTROPY_BANDS], right=False)
    return df


def stacked_label_panel(ax, df, group_col, order):
    ct = pd.crosstab(df[group_col], df["confidence_label"]).reindex(index=order, columns=LABEL_ORDER, fill_value=0)
    frac = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0) * 100
    x = np.arange(len(order))
    bottom = np.zeros(len(order))
    for lab in LABEL_ORDER:
        vals = frac[lab].fillna(0).to_numpy()
        ax.bar(x, vals, bottom=bottom, width=0.72, color=LABEL_COLOR[lab], label=lab, edgecolor=SURFACE, linewidth=2)
        for xi, (v, b) in enumerate(zip(vals, bottom)):
            if v >= 6:
                ax.text(xi, b + v / 2, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
        bottom += vals
    for xi, n in enumerate(ct.sum(axis=1)):
        ax.text(xi, 101.5, f"n={n}", ha="center", va="bottom", fontsize=8, color=INK_2)
    ax.set_ylim(0, 108); ax.grid(axis="y", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
    return ct, frac


ent_df = build_entropy_frame()
print(f"scans analysed: {len(ent_df)}")
print(ent_df)


In [ ]:
qbins = pd.qcut(ent_df["S"], 6, duplicates="drop")
ent_df["S_q"] = qbins
q_order = list(ent_df["S_q"].cat.categories)
q_labels = [f"{c.left:.2f}–{c.right:.2f}" for c in q_order]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9.5, 8), height_ratios=[2.2, 1], sharex=True, gridspec_kw={"hspace": 0.18})
ct_q, frac_q = stacked_label_panel(ax1, ent_df, "S_q", q_order)
ax1.set_ylabel("Share of scans (%)")
ax1.set_title("Confidence label distribution by spectral entropy", fontsize=13, fontweight="bold", loc="left", pad=26)
ax1.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.005), fontsize=9)

x = np.arange(len(q_order))
g = ent_df.groupby("S_q", observed=True)["confidence_score"]
med, q1, q3 = g.median().reindex(q_order), g.quantile(.25).reindex(q_order), g.quantile(.75).reindex(q_order)
ax2.fill_between(x, q1, q3, color="#2a78d6", alpha=0.16, linewidth=0)
ax2.plot(x, med, color="#2a78d6", linewidth=2, marker="o", markersize=7,
         markeredgecolor=SURFACE, markeredgewidth=1.5, label="median score (IQR band)")
ax2.axhline(80, color=INK_2, linewidth=1, linestyle=":", zorder=0)
ax2.set_ylabel("Confidence score")
ax2.set_xlabel("Spectral entropy S (nats), equal-count bins")
ax2.set_xticks(x); ax2.set_xticklabels(q_labels)
ax2.legend(frameon=False, fontsize=9, loc="lower right")
ax2.grid(axis="y", color=GRID, linewidth=0.8); ax2.set_axisbelow(True)
save_and_show(fig, "fig_entropy_vs_confidence")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
mb6 = pd.cut(ent_df["mw"], bins=MW_EDGES, right=False)

for ax, ycol, ylabel, title in [
    (axes[0], "S", "Spectral entropy S (nats)", "MW vs spectral entropy"),
    (axes[1], "eff_peaks", "Effective peaks  (e^S)", "MW vs effective peak count"),
]:
    for lab in LABEL_ORDER:
        s = ent_df[ent_df["confidence_label"] == lab]
        ax.scatter(s["mw"], s[ycol], s=22, alpha=0.5, color=LABEL_COLOR[lab], edgecolor=SURFACE, linewidth=0.4, label=lab)
    lo6 = stats.linregress(ent_df["mw"], ent_df[ycol])
    xs = np.linspace(ent_df["mw"].min(), ent_df["mw"].max(), 100)
    ax.plot(xs, lo6.intercept + lo6.slope * xs, color=INK, linewidth=2, linestyle="--", zorder=5, label="linear fit")
    bmed = ent_df.groupby(mb6, observed=True)[ycol].median()
    bx = ent_df.groupby(mb6, observed=True)["mw"].median()
    ax.plot(bx, bmed, color=INK, linewidth=2.5, marker="o", markersize=8,
            markeredgecolor=SURFACE, markeredgewidth=2, zorder=6, label="median per MW bin")
    r_p, p_p = stats.pearsonr(ent_df["mw"], ent_df[ycol])
    r_s, p_s = stats.spearmanr(ent_df["mw"], ent_df[ycol])
    ax.text(0.02, 0.97, f"Pearson r = {r_p:.2f} (p={p_p:.1e})\nSpearman ρ = {r_s:.2f} (p={p_s:.1e})\nR² = {lo6.rvalue ** 2:.3f}",
            transform=ax.transAxes, va="top", fontsize=9, color=INK_2,
            bbox=dict(boxstyle="round,pad=0.5", facecolor=SURFACE, edgecolor=GRID))
    ax.set_xlabel("Neutral monoisotopic mass (Da)"); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight="bold", loc="left", pad=8)
    ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)

h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, ncol=5, loc="upper left", bbox_to_anchor=(0.06, 1.02), fontsize=9)
save_and_show(fig, "fig_mw_vs_entropy")


In [ ]:
out = []
out.append(f"scans analysed: {len(ent_df)}")
out.append(f"entropy: ms_entropy {DEFAULT_NOISE_THRESHOLD} noise floor, {DEFAULT_MS2_TOLERANCE_DA} Da merge, "
           f"cap {DEFAULT_MAX_PEAK_NUM} peaks, clean_spectrum=True")
out.append("")
out.append("== how the three complexity measures relate ==")
for a, b in [("n_peaks_raw", "n_peaks_filtered"), ("n_peaks_raw", "eff_peaks"),
             ("n_peaks_filtered", "eff_peaks"), ("S", "n_peaks_raw"), ("S", "n_peaks_filtered")]:
    r, p = stats.spearmanr(ent_df[a], ent_df[b])
    out.append(f"  Spearman({a}, {b}) = {r:.3f}  p = {p:.2e}")
out.append("")
out.append(ent_df[["n_peaks_raw", "n_peaks_filtered", "eff_peaks", "S"]].describe().round(2).to_string())
out.append("")
out.append("== each measure vs confidence score (Spearman) ==")
for c in ["n_peaks_raw", "n_peaks_filtered", "eff_peaks", "S", "mw"]:
    r, p = stats.spearmanr(ent_df[c], ent_df["confidence_score"])
    out.append(f"  {c:18s} rho = {r:+.3f}  p = {p:.2e}")
out.append("")
out.append("== confidence by entropy quantile bin ==")
out.append(ct_q.to_string())
out.append("")
out.append(frac_q.round(1).to_string())
out.append("")
out.append(ent_df.groupby("S_q", observed=True)["confidence_score"].agg(["count", "mean", "median"]).round(2).to_string())
out.append("")
out.append("== confidence by named entropy band (bin/mgf_loader._entropy_label) ==")
band_ct = pd.crosstab(ent_df["entropy_band"], ent_df["confidence_label"]).reindex(columns=LABEL_ORDER, fill_value=0)
out.append(band_ct.to_string())
out.append("")
out.append((band_ct.div(band_ct.sum(axis=1).replace(0, np.nan), axis=0) * 100).round(1).to_string())
out.append("")
out.append(ent_df.groupby("entropy_band", observed=True)["confidence_score"].agg(["count", "mean", "median"]).round(2).to_string())
out.append("")
out.append("== MW vs entropy ==")
for c in ["S", "eff_peaks"]:
    r, p = stats.spearmanr(ent_df["mw"], ent_df[c])
    out.append(f"  Spearman(mw, {c}) = {r:+.3f}  p = {p:.2e}")
out.append("")
out.append("== partial: does entropy explain the low-MW penalty? ==")
lowmw = ent_df[ent_df["mw"] < 150]
hi = ent_df[ent_df["mw"] >= 150]
out.append(f"  MW<150:  n={len(lowmw)}  median S={lowmw['S'].median():.3f}  "
           f"median eff_peaks={lowmw['eff_peaks'].median():.1f}  median score={lowmw['confidence_score'].median():.1f}")
out.append(f"  MW>=150: n={len(hi)}  median S={hi['S'].median():.3f}  "
           f"median eff_peaks={hi['eff_peaks'].median():.1f}  median score={hi['confidence_score'].median():.1f}")
u, pu = stats.mannwhitneyu(lowmw["S"], hi["S"])
out.append(f"  Mann-Whitney on S (low vs high MW): U={u:.0f}  p={pu:.2e}")
out.append("")
out.append("  Spearman(mw, score) within each entropy band:")
for band in ent_df["entropy_band"].cat.categories:
    sub = ent_df[ent_df["entropy_band"] == band]
    if len(sub) > 20:
        r, p = stats.spearmanr(sub["mw"], sub["confidence_score"])
        out.append(f"    {band:24s} n={len(sub):4d}  rho={r:+.3f}  p={p:.2e}")

text = "\n".join(out)
open(os.path.join(OUT_REP, "entropy_analysis_stats.txt"), "w").write(text + "\n")
print(text)


## 7. Composite multi-panel figure

Reuses the exact data + drawing logic from §4 and §6:
- **(a)** confidence-label distribution + median-score trend across spectral-entropy bins
- **(b)** precursor MW by confidence label (violin + box + Dunn brackets)
- **(c)** MW vs spectral entropy

In [ ]:
def panel_letter(ax, letter, dx=-0.09, dy=1.04):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=16, fontweight="bold", va="bottom", ha="left", color=INK)


def draw_entropy_panel(ax_bars, ax_line, df):
    df = df.copy()
    df["S_q"] = pd.qcut(df["S"], 6, duplicates="drop")
    q_order = list(df["S_q"].cat.categories)
    q_labels = [f"{c.left:.2f}–{c.right:.2f}" for c in q_order]

    stacked_label_panel(ax_bars, df, "S_q", q_order)
    ax_bars.set_ylabel("Share of scans (%)")
    ax_bars.set_title("Confidence label distribution by spectral entropy", fontsize=12, fontweight="bold", loc="left", pad=24)
    ax_bars.legend(frameon=False, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.005), fontsize=8.5)
    ax_bars.set_xticks(np.arange(len(q_order))); ax_bars.set_xticklabels([])

    x = np.arange(len(q_order))
    g = df.groupby("S_q", observed=True)["confidence_score"]
    med, q1, q3 = g.median().reindex(q_order), g.quantile(.25).reindex(q_order), g.quantile(.75).reindex(q_order)
    ax_line.fill_between(x, q1, q3, color="#2a78d6", alpha=0.16, linewidth=0)
    ax_line.plot(x, med, color="#2a78d6", linewidth=2, marker="o", markersize=6.5,
                 markeredgecolor=SURFACE, markeredgewidth=1.5, label="median score (IQR band)")
    ax_line.axhline(80, color=INK_2, linewidth=1, linestyle=":", zorder=0)
    ax_line.set_ylabel("Confidence score")
    ax_line.set_xlabel("Spectral entropy S (nats), equal-count bins")
    ax_line.set_xticks(x); ax_line.set_xticklabels(q_labels)
    ax_line.legend(frameon=False, fontsize=8.5, loc="lower right")
    ax_line.grid(axis="y", color=GRID, linewidth=0.8); ax_line.set_axisbelow(True)


def draw_violin_panel(ax, df):
    groups = [df.loc[df["confidence_label"] == lab, "mw"].to_numpy() for lab in ORDER]
    pairs = [(0, 1), (1, 2), (0, 2)]
    H_, pH_ = stats.kruskal(*groups)
    eps2_ = epsilon_squared(H_, len(df), len(groups))
    z_ = jonckheere_terpstra(groups)[1]
    dunn_ = sp.posthoc_dunn(df, val_col="mw", group_col="confidence_label", p_adjust="holm").reindex(index=ORDER, columns=ORDER)
    adj_ = np.array([dunn_.loc[ORDER[i], ORDER[j]] for i, j in pairs])

    pos = np.arange(1, len(ORDER) + 1)
    parts = ax.violinplot(groups, positions=pos, widths=0.78, showmeans=False, showmedians=False, showextrema=False)
    for body, lab in zip(parts["bodies"], ORDER):
        body.set_facecolor(LABEL_COLOR[lab]); body.set_alpha(0.32)
        body.set_edgecolor(LABEL_COLOR[lab]); body.set_linewidth(1.5)

    bp = ax.boxplot(groups, positions=pos, widths=0.13, patch_artist=True, showfliers=False,
                    medianprops=dict(color=SURFACE, linewidth=2),
                    whiskerprops=dict(color=INK_2, linewidth=1.2), capprops=dict(color=INK_2, linewidth=1.2))
    for patch, lab in zip(bp["boxes"], ORDER):
        patch.set_facecolor(LABEL_COLOR[lab]); patch.set_edgecolor(LABEL_COLOR[lab])

    rng7 = np.random.default_rng(0)
    for p, gg, lab in zip(pos, groups, ORDER):
        ax.scatter(p + rng7.uniform(-0.06, 0.06, len(gg)), gg, s=4, alpha=0.16, color=LABEL_COLOR[lab], linewidth=0, zorder=1)
        ax.text(p + 0.11, np.median(gg), f"med {np.median(gg):.0f}", va="center", ha="left", fontsize=8, color=INK_2)

    top = max(gg.max() for gg in groups)
    y0, step = top + 60, 85
    for lvl, ((i, j), pa) in enumerate(zip(pairs, adj_)):
        y = y0 + lvl * step
        x1, x2 = pos[i], pos[j]
        ax.plot([x1, x1, x2, x2], [y, y + 22, y + 22, y], color=INK_2, linewidth=1.1)
        star = ("***" if pa < .001 else "**" if pa < .01 else "*" if pa < .05 else "ns")
        ptxt = f"p = {pa:.1e}" if pa < 0.001 else f"p = {pa:.3f}"
        ax.text((x1 + x2) / 2, y + 30, f"{star}   {ptxt}", ha="center", fontsize=8, color=INK_2)
    for p, gg in zip(pos, groups):
        ax.text(p, top + 18, f"n = {len(gg)}", ha="center", fontsize=8.5, color=INK_2)

    ax.set_xlim(0.45, len(ORDER) + 0.72)
    ax.set_ylim(0, y0 + (len(pairs) - 1) * step + 95)
    ax.set_xticks(pos)
    ax.set_xticklabels([l.replace(" evidence", "\nevidence") for l in ORDER])
    ax.set_ylabel("Neutral monoisotopic mass (Da)"); ax.set_xlabel("Confidence label")
    ax.set_title("Precursor mass by confidence label", fontsize=12, fontweight="bold", loc="left", pad=30)
    ax.text(0, 1.008, f"Kruskal-Wallis H = {H_:.1f}, p = {pH_:.1e}, ε² = {eps2_:.3f} (small) · Jonckheere z = {z_:.2f}\n"
                      "brackets: Dunn post-hoc, Holm-adjusted",
            transform=ax.transAxes, fontsize=8.5, color=INK_2, va="bottom")
    ax.grid(axis="y", color=GRID, linewidth=0.8); ax.set_axisbelow(True)


def draw_mw_entropy_panel(ax, df):
    mb = pd.cut(df["mw"], bins=MW_EDGES, right=False)
    for lab in LABEL_ORDER:
        s = df[df["confidence_label"] == lab]
        ax.scatter(s["mw"], s["S"], s=20, alpha=0.5, color=LABEL_COLOR[lab], edgecolor=SURFACE, linewidth=0.4, label=lab)
    lo7 = stats.linregress(df["mw"], df["S"])
    xs = np.linspace(df["mw"].min(), df["mw"].max(), 100)
    ax.plot(xs, lo7.intercept + lo7.slope * xs, color=INK, linewidth=2, linestyle="--", zorder=5, label="linear fit")
    bmed = df.groupby(mb, observed=True)["S"].median()
    bx = df.groupby(mb, observed=True)["mw"].median()
    ax.plot(bx, bmed, color=INK, linewidth=2.5, marker="o", markersize=7.5,
            markeredgecolor=SURFACE, markeredgewidth=2, zorder=6, label="median per MW bin")
    r_p, p_p = stats.pearsonr(df["mw"], df["S"])
    r_s, p_s = stats.spearmanr(df["mw"], df["S"])
    ax.text(0.02, 0.97, f"Pearson r = {r_p:.2f} (p={p_p:.1e})\nSpearman ρ = {r_s:.2f} (p={p_s:.1e})\nR² = {lo7.rvalue ** 2:.3f}",
            transform=ax.transAxes, va="top", fontsize=8.5, color=INK_2,
            bbox=dict(boxstyle="round,pad=0.5", facecolor=SURFACE, edgecolor=GRID))
    ax.set_xlabel("Neutral monoisotopic mass (Da)"); ax.set_ylabel("Spectral entropy S (nats)")
    ax.set_title("MW vs spectral entropy", fontsize=12, fontweight="bold", loc="left", pad=10)
    ax.legend(frameon=True, facecolor=SURFACE, edgecolor=GRID, framealpha=0.9, ncol=1, loc="lower right", fontsize=8)
    ax.grid(color=GRID, linewidth=0.8); ax.set_axisbelow(True)


fig = plt.figure(figsize=(13, 13.5))
outer = fig.add_gridspec(2, 1, height_ratios=[3.1, 2.75], hspace=0.30)
top = outer[0].subgridspec(2, 1, height_ratios=[2.2, 0.95], hspace=0.12)
bot = outer[1].subgridspec(1, 2, width_ratios=[1, 1], wspace=0.24)

ax_a_bars = fig.add_subplot(top[0])
ax_a_line = fig.add_subplot(top[1], sharex=ax_a_bars)
ax_b = fig.add_subplot(bot[0])
ax_c = fig.add_subplot(bot[1])

draw_entropy_panel(ax_a_bars, ax_a_line, ent_df)
draw_violin_panel(ax_b, df4)
draw_mw_entropy_panel(ax_c, ent_df)

panel_letter(ax_a_bars, "a", dx=-0.065, dy=1.14)
panel_letter(ax_b, "b", dx=-0.145, dy=1.10)
panel_letter(ax_c, "c", dx=-0.135, dy=1.10)

save_and_show(fig, "fig_composite")


## 8. `top_compound` vs `top_cosine_compound` comparison report

Flags scans where the multi-metric app annotation (`top_compound`) genuinely differs from
the raw top-cosine hit (`top_cosine_compound`), after normalising away synonym/salt/
collision-energy naming noise, and reports the best raw-cosine hit per scan
(`raw_matches`, sorted by `MQScore`) for cross-reference.

In [ ]:
import unicodedata
from datetime import date

raw_matches = fetch_raw_matches(TASK_ID) if not USE_CACHED_DATA else None
if raw_matches is None:
    # try the repo-root cached full task-result CSV first (kept outside this folder)
    root_csv = os.path.join(REPO_ROOT, f"{TASK_ID}_result.csv")
    if os.path.exists(root_csv):
        print(f"Using cached raw result CSV: {root_csv}")
        raw_matches = pd.read_csv(root_csv)
    else:
        raw_matches = fetch_raw_matches(TASK_ID)

print(f"raw_matches: {len(raw_matches)} rows")


def normalize_name(name):
    if pd.isna(name):
        return ""
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)
    name = re.sub(r"\s*\([^)]*\)\s*$", "", name)
    name = re.sub(r"^spectral match to\s+", "", name)
    name = re.sub(r"[\s_-]*ce\s*[\d.,]+.*$", "", name)
    name = re.sub(r"\s*-\s*[\d.]+\s*ev\s*$", "", name)
    name = name.replace("–", "-").replace("—", "-").replace("_", " ")
    return re.sub(r"\s+", " ", name).strip()


df8 = annex.copy()
df8["top_norm"] = df8["top_compound"].apply(normalize_name)
df8["cosine_norm"] = df8["top_cosine_compound"].apply(normalize_name)
df8["names_differ"] = df8["top_norm"] != df8["cosine_norm"]

raw_top = (
    raw_matches.sort_values("MQScore", ascending=False).groupby("#Scan#").first()
    .reset_index()[["#Scan#", "Compound_Name", "MQScore", "InChIKey"]]
    .rename(columns={"#Scan#": "scan", "Compound_Name": "raw_best_compound",
                      "MQScore": "raw_best_mqscore", "InChIKey": "raw_best_inchikey"})
)
df8 = df8.merge(raw_top, on="scan", how="left")

true_overrides = df8[df8["top_cosine_overridden"] & df8["names_differ"]].copy()
synonym_only = df8[df8["top_cosine_overridden"] & ~df8["names_differ"]].copy()
not_overridden_diff = df8[~df8["top_cosine_overridden"] & df8["names_differ"]]

label_counts = true_overrides["confidence_label"].value_counts()
detail_cols = ["scan", "precursor_mz", "confidence_score", "confidence_label",
               "top_compound", "top_cosine_compound", "max_cosine", "supporting_matches", "total_matches"]
true_overrides_detail = true_overrides[detail_cols].sort_values("confidence_score", ascending=False)


In [ ]:
lines = []
lines.append("=" * 80)
lines.append("TOP-COMPOUND vs TOP-COSINE-COMPOUND COMPARISON REPORT")
lines.append(f"Task: {TASK_ID}")
lines.append(f"Generated: {date.today().isoformat()}")
lines.append("=" * 80)
lines.append("")
lines.append("SUMMARY")
lines.append("-" * 40)
lines.append(f"Total annotated scans:                    {len(df8):>6}")
lines.append(f"Scans where top_cosine_overridden = True: {int(df8['top_cosine_overridden'].sum()):>6}")
lines.append(f"  - Genuinely different compound")
lines.append(f"    (names differ after normalization):    {len(true_overrides):>6}")
lines.append(f"  - Same compound, different label/salt:  {len(synonym_only):>6}")
lines.append(f"Scans where override = False but names")
lines.append(f"  differ (same structure, alt. DB name):  {len(not_overridden_diff):>6}")
lines.append("")
lines.append(f"  Override rate (genuine):  {100 * len(true_overrides) / len(df8):.1f}% of all annotated scans")
lines.append("")
lines.append("CONFIDENCE LABEL BREAKDOWN (genuine overrides only)")
lines.append("-" * 40)
for label, count in label_counts.items():
    pct = 100 * count / len(true_overrides)
    lines.append(f"  {label:<28} {count:>4}  ({pct:.1f}%)")
lines.append("")
lines.append("CONFIDENCE SCORE STATS (genuine overrides)")
lines.append("-" * 40)
stats8 = true_overrides["confidence_score"]
lines.append(f"  Mean:    {stats8.mean():.1f}")
lines.append(f"  Median:  {stats8.median():.1f}")
lines.append(f"  Std:     {stats8.std():.1f}")
lines.append(f"  Min:     {stats8.min():.1f}")
lines.append(f"  Max:     {stats8.max():.1f}")
lines.append("")
lines.append("SYNONYM / SALT CASES (overridden=True, same normalized name)")
lines.append("-" * 40)
for _, row in synonym_only.iterrows():
    lines.append(f"  Scan {row['scan']}: '{row['top_compound']}' vs '{row['top_cosine_compound']}'")
lines.append("")
lines.append("DETAILED TABLE — GENUINE OVERRIDES (sorted by confidence score, desc)")
lines.append("-" * 80)
header = (f"{'Scan':>7}  {'MZ':>8}  {'Score':>6}  {'Label':<22}  {'MaxCosine':>9}  {'Hits':>4}  "
          f"{'TopCompound':<35}  TopCosineCompound")
lines.append(header)
lines.append("-" * 80)
for _, row in true_overrides_detail.iterrows():
    tc = str(row["top_compound"])[:35]
    tcc = str(row["top_cosine_compound"])[:45]
    lbl = str(row["confidence_label"])[:22]
    lines.append(f"{int(row['scan']):>7}  {row['precursor_mz']:>8.3f}  {row['confidence_score']:>6.1f}  "
                 f"{lbl:<22}  {row['max_cosine']:>9.4f}  {int(row['total_matches']):>4}  {tc:<35}  {tcc}")
lines.append("")
lines.append("NOTES ON NORMALIZATION")
lines.append("-" * 40)
lines.append("  Names were normalized before comparison:")
lines.append("  1. Lowercased and Unicode-normalized (NFKD)")
lines.append("  2. Trailing parentheticals stripped (trade names, sources)")
lines.append("  3. Leading 'Spectral match to' prefixes removed")
lines.append("  4. CE (collision energy) annotations stripped (_CE25, - 40.0 eV, etc.)")
lines.append("  5. Underscores replaced with spaces; whitespace collapsed")
lines.append("  The 'top_cosine_overridden' flag in the CSV was also consulted directly.")
lines.append("")
lines.append("=" * 80)

report_text = "\n".join(lines)
out_path = os.path.join(OUT_REP, "compound_comparison_report.txt")
open(out_path, "w").write(report_text)
print(f"Report saved to: {out_path}\n")
print(report_text[:3000])
